# Data quality Assesment

This notebook contains the following list of data quality validation steps:

- Clean feature names
- Detection of missing values 
- Check cardinality
- Validate value ranges 
- Business logic consistency
- Optimize dataframe format for better management if needed

In [31]:
import pandas as pd
import numpy as np

from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
BRONZE_DIR = DATA_DIR / 'bronze'
DATA_FILE = BRONZE_DIR / 'PS_20174392719_1491204439457_log.csv'

In [32]:
df = pd.read_csv(DATA_FILE)
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [33]:
memory_usage = df.memory_usage(deep=True).sum()
print(f"Memory size: {memory_usage / (1024 ** 2):.2f} MB")

Memory size: 1598.19 MB


**Observations:**

The df is relatively big (~ 1.56 GB) and my computer is not the best so I will:
- use parquet instead of csv format to manage the dfs
- save plots as images 
- save heavy and slow functions inside the cache

## Clean feature names

CamelCase detected, let's change it to snake_case and lowercase only

In [34]:
data = df.copy()

rename_map = {
        'oldbalanceOrg': 'old_balance_orig',
        'newbalanceOrig': 'new_balance_orig',
        'oldbalanceDest': 'old_balance_dest',
        'newbalanceDest': 'new_balance_dest',
        'nameOrig': 'name_orig',
        'nameDest': 'name_dest',
        'isFraud': 'is_fraud',
        'isFlaggedFraud': 'is_flagged_fraud'
    }
    
rename_map = {k: v for k, v in rename_map.items() if k in data.columns}
data = data.rename(columns=rename_map)

if rename_map:
    print(f"Renamed {len(rename_map)} columns")
    for old, new in rename_map.items():
        print(f"{old} -> {new}")

else:
    print(" No columns to rename")

Renamed 8 columns
oldbalanceOrg -> old_balance_orig
newbalanceOrig -> new_balance_orig
oldbalanceDest -> old_balance_dest
newbalanceDest -> new_balance_dest
nameOrig -> name_orig
nameDest -> name_dest
isFraud -> is_fraud
isFlaggedFraud -> is_flagged_fraud


## Missing values

In [35]:
data.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   step              6362620 non-null  int64  
 1   type              6362620 non-null  object 
 2   amount            6362620 non-null  float64
 3   name_orig         6362620 non-null  object 
 4   old_balance_orig  6362620 non-null  float64
 5   new_balance_orig  6362620 non-null  float64
 6   name_dest         6362620 non-null  object 
 7   old_balance_dest  6362620 non-null  float64
 8   new_balance_dest  6362620 non-null  float64
 9   is_fraud          6362620 non-null  int64  
 10  is_flagged_fraud  6362620 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


**Observations**: 
- 11 columns
- 6362620 rows
- No missing values!
- dtypes: 3 int64, 3 object, 5 float64

## Cardinality

In [36]:
for col in data.columns:
    unique_count = data[col].nunique()
    total_count = len(data)
    cardinality_ratio = unique_count / total_count
    
    if data[col].dtype in ['object']:
        var_type = 'Categorical'
        if unique_count <= 10:
            action = "Low" # these are good for encoding
        elif unique_count <= 50:
            action = "Medium" # grouping
        elif unique_count <= 1000:
            action = "High" # aggregate
        else:
            action = "Very High" # aggregate or drop
    else:
        var_type = 'Numerical'
        if cardinality_ratio < 0.01:
            action = "Low variance" # potentially ugly, ask yourself why
        else:
            action = "Normal" 
    
    print(f"{col:<20} {var_type:<12} {unique_count:>15,} {cardinality_ratio:>11.2%} {action:<25}")


step                 Numerical                743       0.01% Low variance             
type                 Categorical                5       0.00% Low                      
amount               Numerical          5,316,900      83.56% Normal                   
name_orig            Categorical        6,353,307      99.85% Very High                
old_balance_orig     Numerical          1,845,844      29.01% Normal                   
new_balance_orig     Numerical          2,682,586      42.16% Normal                   
name_dest            Categorical        2,722,362      42.79% Very High                
old_balance_dest     Numerical          3,614,697      56.81% Normal                   
new_balance_dest     Numerical          3,555,499      55.88% Normal                   
is_fraud             Numerical                  2       0.00% Low variance             
is_flagged_fraud     Numerical                  2       0.00% Low variance             


**Obervations:**

- step: Time step in hours of the month (1-743). Create new derived features later like step_day, step_hour, is_weekend or is_night
- type: Transaction type. Keep OHT or ordinal encoding
- amount: Continuous numeric feature. Maybe scale but let's see the distribution first
- name_orig: Extremely high, probably a unique identifier for each sender (account ID). Drop or use for aggregation (maybe prefix of origin or frequency). Also maybe hash encoding? Check on bivar
- old_balance_orig, new_balance_orig, old_balance_dest, new_balance_dest: Continuous numeric. Keep to check the balances later. Maybe they show some weird behaviour?
- name_dest: High-cardinality, same procedure as name_orig

- is_fraud: target
- is_flagged_fraud: target too

## Negative values

In [37]:
neg_vals = []
for col in data.select_dtypes(include=[np.number]).columns:
    if (data[col] < 0).any():
        n_negative = (data[col] < 0).sum()
        neg_vals.append(f" This {col} has {n_negative:,} negative values")

if neg_vals:
    for val in neg_vals:
        print(neg_vals)
else:
    print("No obvious negative values detected")
    print("Great!")

No obvious negative values detected
Great!


## Business logic consistency

Here I'm going to check different assumptions that came to my mind to ensure that the data follows real-world business rules.

### Assumption 1: Value Range Validation

- Transaction amounts must be > 0
- Balances must be >= 0
- Step must be positive

Negative amounts or balances indicate data corruption or processing errors.


In [38]:
validation_set = data[['step', 
           'type', 
           'amount', 
           'old_balance_orig', 
           'new_balance_orig', 
           'old_balance_dest', 
           'new_balance_dest', 
           'is_fraud']].copy()

invalid_zero_negative = validation_set.query("amount <= 0")
print(f"Negative/zero amounts: {len(invalid_zero_negative)} transactions")
invalid_zero_negative

Negative/zero amounts: 16 transactions


,step,type,amount,old_balance_orig,new_balance_orig,old_balance_dest,new_balance_dest,is_fraud
2736447,212,CASH_OUT,0.0,0.0,0.0,0.00,0.00,1
3247298,250,CASH_OUT,0.0,0.0,0.0,0.00,0.00,1
3760289,279,CASH_OUT,0.0,0.0,0.0,538547.63,538547.63,1
5563714,387,CASH_OUT,0.0,0.0,0.0,7970766.57,7970766.57,1
5996408,425,CASH_OUT,0.0,0.0,0.0,76759.90,76759.90,1
5996410,425,CASH_OUT,0.0,0.0,0.0,2921531.34,2921531.34,1
6168500,554,CASH_OUT,0.0,0.0,0.0,230289.66,230289.66,1
6205440,586,CASH_OUT,0.0,0.0,0.0,1328472.86,1328472.86,1
6266414,617,CASH_OUT,0.0,0.0,0.0,0.00,0.00,1
6281483,646,CASH_OUT,0.0,0.0,0.0,0.00,0.00,1


Fraudsters tried to withdraw money but couldn't. 

It would be interesting to do further research on these transactions to check if those same accounts are reused, which hours they were made, etc.

In [39]:
balance_cols = ['old_balance_orig',	'new_balance_orig',	'old_balance_dest',	'new_balance_dest']

for col in balance_cols:
    invalid_balance = validation_set[col] < 0
    print(f"  {col}:")
    print(f" Total valid: {(~invalid_balance).sum():,} ({(~invalid_balance).mean()*100:.2f}%)")
    print(f" Total invalid: {invalid_balance.sum():,} ({invalid_balance.mean()*100:.2f}%)""\n")

  old_balance_orig:
 Total valid: 6,362,620 (100.00%)
 Total invalid: 0 (0.00%)

  new_balance_orig:
 Total valid: 6,362,620 (100.00%)
 Total invalid: 0 (0.00%)

  old_balance_dest:
 Total valid: 6,362,620 (100.00%)
 Total invalid: 0 (0.00%)

  new_balance_dest:
 Total valid: 6,362,620 (100.00%)
 Total invalid: 0 (0.00%)



In [40]:
invalid_step = validation_set['step'] <= 0
print(f" Time Step Validation:")
print(f" Total valid: {(~invalid_step).sum():,} ({(~invalid_step).mean()*100:.2f}%)")
print(f" Total invalid: {invalid_step.sum():,} ({invalid_step.mean()*100:.2f}%)")

 Time Step Validation:
 Total valid: 6,362,620 (100.00%)
 Total invalid: 0 (0.00%)


### Assumption 2: Balance Consistency

I assume that for origin and destination, balance arithmetic should be consistent, with this I mean that:

``` old_balance_orig - amount = new_balance_orig ```

for transfer, cash_out and debit

&

``` old_balance_dest + amount = new_balance_dest```

for transfer and cash_in



In [41]:
# Let's give 1 cent tolerance for rounding errors
tolerance = 0.01  

print(f"Origin Balance Consistency:")

deduct_types = ['TRANSFER', 'CASH_OUT', 'PAYMENT', 'DEBIT']
deduct_mask = validation_set['type'].isin(deduct_types)
expected_new_balance_orig = validation_set.loc[deduct_mask, 'old_balance_orig'] - validation_set.loc[deduct_mask, 'amount']
actual_new_balance_orig = validation_set.loc[deduct_mask, 'new_balance_orig']

balance_diff = np.abs(expected_new_balance_orig - actual_new_balance_orig)
inconsistent_orig = balance_diff > tolerance

print(f"  Total transactions checked: {deduct_mask.sum():,} ({', '.join(deduct_types)})")
print(f" Total consistent: {(~inconsistent_orig).sum():,} ({(~inconsistent_orig).mean()*100:.2f}%)")
print(f" Total inconsistent: {inconsistent_orig.sum():,} ({inconsistent_orig.mean()*100:.2f}%)")
print(f" Max difference: {balance_diff.max():.2f} USD")
print(f" Mean difference: {balance_diff.mean():.2f} USD")

Origin Balance Consistency:
  Total transactions checked: 4,963,336 (TRANSFER, CASH_OUT, PAYMENT, DEBIT)
 Total consistent: 1,284,929 (25.89%)
 Total inconsistent: 3,678,407 (74.11%)
 Max difference: 92445516.64 USD
 Mean difference: 162541.33 USD


This type of errors can be produced by processing errors, missing inmediate transactions, extra fees, etc.

As next steps I would investigate if there are fees, option for credit in transfer and payment, etc.

In [42]:
add_types = ['TRANSFER', 'CASH_IN']
add_mask = validation_set['type'].isin(add_types)

expected_new_balance_dest = validation_set.loc[add_mask, 'old_balance_dest'] + validation_set.loc[add_mask, 'amount']
actual_new_balance_dest = validation_set.loc[add_mask, 'new_balance_dest']

balance_diff_dest = np.abs(expected_new_balance_dest - actual_new_balance_dest)
inconsistent_dest = balance_diff_dest > tolerance

print(f"  Total transactions checked: {add_mask.sum():,} ({', '.join(add_types)})")
print(f" Total Consistent: {(~inconsistent_dest).sum():,} ({(~inconsistent_dest).mean()*100:.2f}%)")
print(f" Total Inconsistent: {inconsistent_dest.sum():,} ({inconsistent_dest.mean()*100:.2f}%)")
print(f" Max difference: ${balance_diff_dest.max():.2f}")
print(f" Mean difference: ${balance_diff_dest.mean():.2f}")

  Total transactions checked: 1,932,193 (TRANSFER, CASH_IN)
 Total Consistent: 412,098 (21.33%)
 Total Inconsistent: 1,520,095 (78.67%)
 Max difference: $75885725.63
 Mean difference: $258095.95


Same thing. We have to fully understand business cases, then we can discard that these transactions don't have errors or we lack information to understand them.

### Assumption 3: Insufficient funds

Amounts exceed available balance in origin?




In [43]:
deduct_types = ['TRANSFER', 'CASH_OUT', 'PAYMENT', 'DEBIT']
deduct_mask = validation_set['type'].isin(deduct_types)

insufficient_funds = validation_set.loc[deduct_mask, 'amount'] > validation_set.loc[deduct_mask, 'old_balance_orig']

print(f" Total Transactions checked: {deduct_mask.sum():,} ({', '.join(deduct_types)})")
print(f" Sufficient funds: {(~insufficient_funds).sum():,} ({(~insufficient_funds).mean()*100:.2f}%)")
print(f" Insufficient funds: {insufficient_funds.sum():,} ({insufficient_funds.mean()*100:.2f}%)")

 Total Transactions checked: 4,963,336 (TRANSFER, CASH_OUT, PAYMENT, DEBIT)
 Sufficient funds: 1,361,792 (27.44%)
 Insufficient funds: 3,601,544 (72.56%)


Mmmhh... very interesting. Does it correlate with fraud? Again we need to further check those values with business related information

### Assumption 4: names on origin and destination match the types of transactions

In [44]:
print(f" Name types in the ORIGIN: {data['name_orig'].str[0].value_counts().to_dict()}")
print(f" Name types in the DESTINATION: {data['name_dest'].str[0].value_counts().to_dict()}")


 Name types in the ORIGIN: {'C': 6362620}
 Name types in the DESTINATION: {'C': 4211125, 'M': 2151495}



Account names start with C (Customer) or M (Merchant). I assume that every transaction can have specific name codes based on their nature: 
- PAYMENT: Origin = Customer (C), Destination = Merchant (M)
- TRANSFER: Origin = Customer (C), Destination = Customer (C)
- CASH_OUT: Origin = Customer (C), Destination can vary
- CASH_IN: Money coming in (destination balance increases)
- DEBIT: Origin = Customer (C)

In [45]:
name_set = data[['type',
                 'name_orig',
                 'name_dest']].copy()

name_set['orig_prefix'] = name_set['name_orig'].str[0]
name_set['dest_prefix'] = name_set['name_dest'].str[0]

# Check the previous rules

rules_validation = {
    'PAYMENT': {
        'mask': name_set['type'] == 'PAYMENT',
        'valid': (name_set['orig_prefix'] == 'C') & (name_set['dest_prefix'] == 'M')
    },
    'TRANSFER': {
        'mask': name_set['type'] == 'TRANSFER',
        'valid': (name_set['orig_prefix'] == 'C') & (name_set['dest_prefix'] == 'C')
    },
    'CASH_OUT': {
        'mask': name_set['type'] == 'CASH_OUT',
        'valid': name_set['orig_prefix'] == 'C'
    },
    'CASH_IN': {
        'mask': name_set['type'] == 'CASH_IN',
        'valid': name_set['dest_prefix'] == 'C'
    },
    'DEBIT': {
        'mask': name_set['type'] == 'DEBIT',
        'valid': name_set['orig_prefix'] == 'C'
    }
}

total_violations = 0
for trans_type, rule_info in rules_validation.items():
    if rule_info['mask'].sum() > 0:
        mask = rule_info['mask']
        valid = rule_info['valid'][mask]
        
        print(f"\n  {trans_type}:")
        print(f"    ✓ Compliant: {valid.sum():,} ({valid.mean()*100:.2f}%)")
        print(f"    ✗ Violations: {(~valid).sum():,} ({(~valid).mean()*100:.2f}%)")
        
        total_violations += (~valid).sum()


  PAYMENT:
    ✓ Compliant: 2,151,495 (100.00%)
    ✗ Violations: 0 (0.00%)

  TRANSFER:
    ✓ Compliant: 532,909 (100.00%)
    ✗ Violations: 0 (0.00%)

  CASH_OUT:
    ✓ Compliant: 2,237,500 (100.00%)
    ✗ Violations: 0 (0.00%)

  CASH_IN:
    ✓ Compliant: 1,399,284 (100.00%)
    ✗ Violations: 0 (0.00%)

  DEBIT:
    ✓ Compliant: 41,432 (100.00%)
    ✗ Violations: 0 (0.00%)


### Assumption 5: temporal consistency

Transaction steps have to be 
- Sequential (increases)
- No duplicate transactions at the same time, how to find them? They must share step, name in the origin, amount and type
- The range of the values shouls be from 1 to 743 = 1 month



In [46]:
min_step = data['step'].min()
max_step = data['step'].max()


print(f"  Minimum step: {min_step}")
print(f"  Maximum step: {max_step}")

duplicates_check = data.groupby(['step', 'name_orig', 'amount', 'type']).size()
duplicates = duplicates_check[duplicates_check > 1]

print(f"  Total unique transaction groups: {len(duplicates_check):,}")
print(f"  Duplicate transaction groups: {len(duplicates):,}")

  Minimum step: 1
  Maximum step: 743
  Total unique transaction groups: 6,362,620
  Duplicate transaction groups: 0


In [47]:
txns_per_step = data.groupby('step').size()

print(f"  Total time steps: {len(txns_per_step)}")
print(f"  Average transactions per step: {txns_per_step.mean():.0f}")
print(f"  Max transactions in single step: {txns_per_step.max():,}")
print(f"  Min transactions in single step: {txns_per_step.min():,}")

# Check if there are any gaps in time steps
expected_steps = set(range(min_step, max_step + 1))
actual_steps = set(data['step'].unique())
missing_steps = expected_steps - actual_steps


if len(missing_steps) > 0:
    print(f"\n Missing time steps: {len(missing_steps)} steps with no transactions")
    print(f" First few missing: {sorted(list(missing_steps))[:10]}")
else:
    print(f"\n All time steps have transactions (no gaps)")


  Total time steps: 743
  Average transactions per step: 8563
  Max transactions in single step: 51,352
  Min transactions in single step: 2

 All time steps have transactions (no gaps)


### Assumption 6: Integrity in the data

Features should follow different formats and ranges. Here I'm going to check few of them:

- Type has to belong to the following categories: payment, transfer, cash_out, cash_in and debit
- Fraud label must be binary: fraud and no_fraud
- 

In [48]:
valid_types = ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'CASH_IN', 'DEBIT']
actual_types = data['type'].unique()

print(f"  Expected types: {valid_types}")
print(f"  Actual types found: {list(actual_types)}")

  Expected types: ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'CASH_IN', 'DEBIT']
  Actual types found: ['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN']


In [49]:

if 'is_fraud' in data.columns:
    valid_fraud_labels = [0, 1]
    actual_fraud_labels = data['is_fraud'].unique()
    
    print(f"  Expected labels: {valid_fraud_labels}")
    print(f"  Actual labels: {list(actual_fraud_labels)}")

  Expected labels: [0, 1]
  Actual labels: [0, 1]


### Assumption 7: Consistency in the amounts

Currency amounts should have max 2 decimal places


In [50]:
amount_str = data["amount"].astype(str)

invalid_decimals = amount_str.str.contains(r'\.\d{3,}')

print(f"Invalid decimal precision: {invalid_decimals.sum()} transactions")
print(f"Valid amounts: {(~invalid_decimals).sum()} transactions")

invalid_rows = data[invalid_decimals]
invalid_rows

Invalid decimal precision: 0 transactions
Valid amounts: 6362620 transactions


,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud


## Optimize data types

Reasoning behind the following line:
- Step to int16: it it goes from 1 to 743. I assume that the df is created monthly and we will have the first approach using this time range. In case this changes and we start having more data to work with that needs a bigger range than int16 (-32,768 to 32,767 values) we should increase the data type to int32.

- Type, nameOrig, nameDest and isFraud to category: for better memory usage.

- Amount, oldbalanceOrig, newbalanceOrig, oldbalanceDest and newbalanceDest to float32: they all go from 0.0 to a maximum of 10^8. I assume that the all the balances cannot be much more bigger (i.e. assuming business knowledge related to type of transactions) so the range of float32 (±3.4 × 10^38) would be enough.

In [51]:
data

,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


In [52]:
memory_usage = data.memory_usage(deep=True).sum()
print(f"Memory size: {memory_usage / (1024 ** 2):.2f} MB")

Memory size: 1598.19 MB


In [53]:
data[['step']] = data[['step']].astype('int16')
data[['type', 'name_orig', 'name_dest', 'is_fraud', 'is_flagged_fraud']] = data[['type', 'name_orig', 'name_dest', 'is_fraud', 'is_flagged_fraud']].astype('category')
data[['amount', 'old_balance_orig', 'new_balance_orig', 'old_balance_dest', 'new_balance_dest']] = data[['amount', 'old_balance_orig', 'new_balance_orig', 'old_balance_dest', 'new_balance_dest']].astype('float32')


## Check memory size after modifications



In [54]:
memory_usage = data.memory_usage(deep=True).sum()
print(f"Memory size: {memory_usage / (1024 ** 2):.2f} MB")

Memory size: 977.82 MB


happy that I went from 1.65 GB to  998 MB 

In [57]:
data.to_parquet('data/silver/df_fraud.parquet', index=False, engine='fastparquet')

In [56]:
data

,step,type,amount,name_orig,old_balance_orig,new_balance_orig,name_dest,old_balance_dest,new_balance_dest,is_fraud,is_flagged_fraud
0,1,PAYMENT,9.839640e+03,C1231006815,170136.000,160296.359375,M1979787155,0.000000e+00,0.000,0,0
1,1,PAYMENT,1.864280e+03,C1666544295,21249.000,19384.720703,M2044282225,0.000000e+00,0.000,0,0
2,1,TRANSFER,1.810000e+02,C1305486145,181.000,0.000000,C553264065,0.000000e+00,0.000,1,0
3,1,CASH_OUT,1.810000e+02,C840083671,181.000,0.000000,C38997010,2.118200e+04,0.000,1,0
4,1,PAYMENT,1.166814e+04,C2048537720,41554.000,29885.859375,M1230701703,0.000000e+00,0.000,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,3.396821e+05,C786484425,339682.125,0.000000,C776919290,0.000000e+00,339682.125,1,0
6362616,743,TRANSFER,6.311410e+06,C1529008245,6311409.500,0.000000,C1881841831,0.000000e+00,0.000,1,0
6362617,743,CASH_OUT,6.311410e+06,C1162922333,6311409.500,0.000000,C1365125890,6.848884e+04,6379898.000,1,0
6362618,743,TRANSFER,8.500025e+05,C1685995037,850002.500,0.000000,C2080388513,0.000000e+00,0.000,1,0
